In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

try:
    from IPython.display import display, HTML
except Exception:
    display = None
    HTML = None

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


LEFT_EYE_IDX = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
RIGHT_EYE_IDX = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
MOUTH_IDX_FULL = [
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291,
    78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    185, 40, 39, 37, 0, 267, 269, 270, 409,
    191, 80, 81, 82, 13, 312, 311, 310, 415,
]


@dataclass
class Config:
    model_path: str = "/content/face_landmarker.task"
    root_dir: str = "/kaggle/input/dddeeee"
    save_root: str = "/content/output"

    region: str = "face"   # face | left_eye | right_eye | mouth
    split: str = "train"    # train | validation | test

    padding_ratio: float = 0.10
    use_live_display: bool = True
    fail_if_label_missing: bool = True

    # Canonical names used in labels / output folders.
    normalize_output_condition: bool = True


class DisplayManager:
    def __init__(self, enabled: bool = True):
        self.enabled = enabled and display is not None and HTML is not None
        self.live_handle = None
        self.summary_handle = None
        self.all_summaries: List[str] = []

    def init(self):
        self.all_summaries = []
        if not self.enabled:
            return
        self.live_handle = display(HTML("<pre>[LIVE] Waiting to start...</pre>"), display_id=True)
        self.summary_handle = display(HTML("<pre>[SUMMARY] No completed videos yet.</pre>"), display_id=True)

    def update_live(self, lines: List[str]):
        text = "\n".join(lines)
        if self.enabled and self.live_handle is not None:
            self.live_handle.update(HTML(f"<pre>{text}</pre>"))
        else:
            print(text)

    def append_summary(self, lines: List[str]):
        text = "\n".join(lines)
        self.all_summaries.append(text)
        if self.enabled and self.summary_handle is not None:
            combined = "\n\n".join(self.all_summaries)
            self.summary_handle.update(HTML(f"<pre>{combined}</pre>"))
        else:
            print(text)

    def clear_live(self):
        if self.enabled and self.live_handle is not None:
            self.live_handle.update(HTML("<pre></pre>"))


def normalize_condition(condition: str) -> str:
    condition = condition.strip().lower()
    mapping = {
        "nightnoglasses": "night_noglasses",
        "night_no_glasses": "night_noglasses",
        "night-noglasses": "night_noglasses",
        "night_noglasses": "night_noglasses",
        "noglasses": "noglasses",
        "no_glasses": "noglasses",
        "nightglasses": "nightglasses",
        "glasses": "glasses",
        "sunglasses": "sunglasses",
    }
    return mapping.get(condition, condition)


def label_suffix_for_region(region: str) -> str:
    region = region.lower()
    mapping = {
        "face": "head",
        "left_eye": "eye",
        "right_eye": "eye",
        "mouth": "mouth",
    }
    if region not in mapping:
        raise ValueError(f"Unsupported region: {region}")
    return mapping[region]


def create_face_landmarker(model_path: str):
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")

    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.2,
        min_face_presence_confidence=0.2,
        min_tracking_confidence=0.2,
    )
    return vision.FaceLandmarker.create_from_options(options)


def load_labels(label_path: str) -> List[int]:
    if not os.path.isfile(label_path):
        raise FileNotFoundError(f"Label file not found: {label_path}")

    with open(label_path, "r", encoding="utf-8") as f:
        content = f.read().strip()

    return [int(ch) for ch in content if ch.isdigit()]


def crop_with_padding(frame: np.ndarray, x1: int, y1: int, x2: int, y2: int,
    out_h: int, out_w: int, fill_value: int = 0) -> np.ndarray:
    h, w = frame.shape[:2]
    crop = np.full((out_h, out_w, 3), fill_value, dtype=frame.dtype)

    src_x1 = max(0, x1)
    src_y1 = max(0, y1)
    src_x2 = min(w, x2)
    src_y2 = min(h, y2)

    copy_w = src_x2 - src_x1
    copy_h = src_y2 - src_y1
    if copy_w <= 0 or copy_h <= 0:
        return crop

    dst_x1 = src_x1 - x1
    dst_y1 = src_y1 - y1
    dst_x2 = dst_x1 + copy_w
    dst_y2 = dst_y1 + copy_h

    crop[dst_y1:dst_y2, dst_x1:dst_x2] = frame[src_y1:src_y2, src_x1:src_x2]
    return crop


def get_square_box_from_points(points: List[Tuple[float, float]], padding_ratio: float) -> Tuple[int, int, int, int, int]:
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    box_w = x_max - x_min
    box_h = y_max - y_min
    side = max(box_w, box_h)
    final_side = max(int(round(side * (1 + 2 * padding_ratio))), 1)

    cx = (x_min + x_max) / 2.0
    cy = (y_min + y_max) / 2.0

    x1 = int(round(cx - final_side / 2.0))
    y1 = int(round(cy - final_side / 2.0))
    x2 = x1 + final_side
    y2 = y1 + final_side
    return x1, y1, x2, y2, final_side


def get_face_box(landmarks, frame_w: int, frame_h: int, padding_ratio: float):
    points = [(lm.x * frame_w, lm.y * frame_h) for lm in landmarks]
    return get_square_box_from_points(points, padding_ratio)


def get_eye_box(landmarks, frame_w: int, frame_h: int, eye_indices: List[int], padding_ratio: float):
    points = [(landmarks[i].x * frame_w, landmarks[i].y * frame_h) for i in eye_indices]
    return get_square_box_from_points(points, padding_ratio)


def get_region_box(landmarks, frame_w: int, frame_h: int, region_indices: List[int]):
    xs = [int(landmarks[i].x * frame_w) for i in region_indices]
    ys = [int(landmarks[i].y * frame_h) for i in region_indices]

    x1 = max(min(xs), 0)
    y1 = max(min(ys), 0)
    x2 = min(max(xs) + 1, frame_w)
    y2 = min(max(ys) + 1, frame_h)
    return x1, y1, x2, y2


class CropStrategy:
    def __init__(self, region: str, padding_ratio: float):
        self.region = region
        self.padding_ratio = padding_ratio
        self.prev_box = None

    def extract(self, frame: np.ndarray, landmarks) -> Tuple[Optional[np.ndarray], bool, bool]:
        h, w = frame.shape[:2]

        if self.region == "face":
            if landmarks is not None:
                x1, y1, x2, y2, side = get_face_box(landmarks, w, h, self.padding_ratio)
                self.prev_box = (x1, y1, x2, y2, side)
                return crop_with_padding(frame, x1, y1, x2, y2, side, side, 0), False, False
            if self.prev_box is not None:
                x1, y1, x2, y2, side = self.prev_box
                return crop_with_padding(frame, x1, y1, x2, y2, side, side, 0), True, False
            side = min(h, w)
            x1 = (w - side) // 2
            y1 = (h - side) // 2
            x2 = x1 + side
            y2 = y1 + side
            self.prev_box = (x1, y1, x2, y2, side)
            return crop_with_padding(frame, x1, y1, x2, y2, side, side, 0), True, False

        if self.region == "left_eye":
            eye_idx = LEFT_EYE_IDX
        elif self.region == "right_eye":
            eye_idx = RIGHT_EYE_IDX
        else:
            eye_idx = None

        if eye_idx is not None:
            if landmarks is not None:
                x1, y1, x2, y2, side = get_eye_box(landmarks, w, h, eye_idx, self.padding_ratio)
                self.prev_box = (x1, y1, x2, y2, side)
                return crop_with_padding(frame, x1, y1, x2, y2, side, side, 0), False, False
            if self.prev_box is not None:
                x1, y1, x2, y2, side = self.prev_box
                return crop_with_padding(frame, x1, y1, x2, y2, side, side, 0), True, False
            return None, False, True

        if self.region == "mouth":
            if landmarks is None:
                return None, False, True
            x1, y1, x2, y2 = get_region_box(landmarks, w, h, MOUTH_IDX_FULL)
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0 or crop.shape[0] <= 0 or crop.shape[1] <= 0:
                return None, False, True
            return crop, False, False

        raise ValueError(f"Unsupported region: {self.region}")


def parse_validation_or_test_video_name(video_stem: str) -> Tuple[str, str, str]:
    parts = video_stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Invalid filename format: {video_stem}")

    person_id = parts[0]
    state = parts[-1]
    raw_condition = "_".join(parts[1:-1])
    condition = normalize_condition(raw_condition)
    return person_id, condition, state


def process_video(video_path: str, label_path: str, output_dir: str, model_path: str,
    save_prefix: str, region: str, padding_ratio: float, display_name: Optional[str],
    display_mgr: DisplayManager) -> None:
    if not os.path.isfile(video_path):
        raise FileNotFoundError(f"Video file not found: {video_path}")

    labels = load_labels(label_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    reported_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        fps = 30.0

    landmarker = create_face_landmarker(model_path)
    strategy = CropStrategy(region=region, padding_ratio=padding_ratio)

    frame_idx = 0
    saved_count = 0
    missed_count = 0
    fallback_used = 0
    discarded_extra_frames = 0
    shown_name = display_name if display_name is not None else os.path.basename(video_path)

    print("=" * 72)
    print(f"[INFO] Region           : {region}")
    print(f"[INFO] Processing video : {video_path}")
    print(f"[INFO] Label file       : {label_path}")
    print(f"[INFO] Save folder      : {output_dir}")
    print(f"[INFO] Save prefix      : {save_prefix}")
    print(f"[INFO] Reported frames  : {reported_total_frames}")
    print(f"[INFO] Total labels     : {len(labels)}")
    print("=" * 72)

    try:
        while frame_idx < len(labels):
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int((frame_idx / fps) * 1000)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            landmarks = result.face_landmarks[0] if result.face_landmarks else None

            crop, used_fallback, missed = strategy.extract(frame, landmarks)
            if used_fallback:
                fallback_used += 1
            if missed or crop is None:
                missed_count += 1
            else:
                label = labels[frame_idx]
                save_name = f"{save_prefix}_{frame_idx:06d}_{label}.png"
                save_path = os.path.join(output_dir, save_name)
                ok = cv2.imwrite(save_path, crop)
                if not ok:
                    raise RuntimeError(f"Failed to save image: {save_path}")
                saved_count += 1

            total_for_display = max(reported_total_frames, len(labels))
            live_lines = [
                "=" * 72,
                f"[LIVE] Video               : {shown_name}",
                f"[LIVE] Region              : {region}",
                f"[LIVE] Frame               : {frame_idx + 1}/{total_for_display}",
                f"[LIVE] Saved               : {saved_count}",
                f"[LIVE] Missed detect       : {missed_count}",
                f"[LIVE] Fallback used       : {fallback_used}",
                "=" * 72,
            ]
            display_mgr.update_live(live_lines)
            frame_idx += 1

        while True:
            ret, _ = cap.read()
            if not ret:
                break
            discarded_extra_frames += 1

    finally:
        cap.release()
        landmarker.close()

    actual_frames_read = frame_idx + discarded_extra_frames
    diff = len(labels) - actual_frames_read
    summary_lines = [
        "=" * 72,
        f"[SUMMARY] Video               : {shown_name}",
        f"[SUMMARY] Region              : {region}",
        f"[SUMMARY] Frames read         : {actual_frames_read}",
        f"[SUMMARY] Labels              : {len(labels)}",
        f"[SUMMARY] Labels used         : {frame_idx}",
        f"[SUMMARY] Discarded frames    : {discarded_extra_frames}",
    ]

    if diff == 0:
        summary_lines += [
            "[SUMMARY] Status              : PERFECT MATCH ✅",
            "[SUMMARY] Extra labels        : 0",
            "[SUMMARY] Extra frames        : 0",
        ]
    elif diff > 0:
        summary_lines += [
            "[SUMMARY] Status              : EXTRA LABELS ⚠️",
            f"[SUMMARY] Extra labels        : {diff}",
            "[SUMMARY] Extra frames        : 0",
        ]
    else:
        summary_lines += [
            "[SUMMARY] Status              : EXTRA FRAMES DISCARDED ⚠️",
            "[SUMMARY] Extra labels        : 0",
            f"[SUMMARY] Extra frames        : {-diff}",
        ]

    summary_lines += [
        f"[SUMMARY] Saved               : {saved_count}",
        f"[SUMMARY] Missed detect       : {missed_count}",
        f"[SUMMARY] Fallback used       : {fallback_used}",
        f"[SUMMARY] Output folder       : {output_dir}",
        "=" * 72,
    ]
    display_mgr.append_summary(summary_lines)


def build_output_condition(condition: str, cfg: Config) -> str:
    return normalize_condition(condition) if cfg.normalize_output_condition else condition


def process_train(cfg: Config, display_mgr: DisplayManager) -> None:
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    suffix = label_suffix_for_region(cfg.region)

    for person_entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        person_id = person_entry.name
        for condition_entry in sorted(os.scandir(person_entry.path), key=lambda e: e.name):
            if not condition_entry.is_dir():
                continue

            raw_condition = condition_entry.name
            out_condition = build_output_condition(raw_condition, cfg)

            for file_entry in sorted(os.scandir(condition_entry.path), key=lambda e: e.name):
                if not file_entry.is_file() or not file_entry.name.lower().endswith(".avi"):
                    continue

                behavior = os.path.splitext(file_entry.name)[0]
                video_path = file_entry.path
                label_path = os.path.join(condition_entry.path, f"{person_id}_{behavior}_{suffix}.txt")

                if not os.path.isfile(label_path):
                    msg = f"Missing train label: {label_path}"
                    if cfg.fail_if_label_missing:
                        raise FileNotFoundError(msg)
                    print(f"[SKIP] {msg}")
                    continue

                output_dir = os.path.join(cfg.save_root, person_id, out_condition, behavior)
                os.makedirs(output_dir, exist_ok=True)
                save_prefix = f"{person_id}_{out_condition}_{behavior}"

                process_video(
                    video_path=video_path,
                    label_path=label_path,
                    output_dir=output_dir,
                    model_path=cfg.model_path,
                    save_prefix=save_prefix,
                    region=cfg.region,
                    padding_ratio=cfg.padding_ratio,
                    display_name=behavior,
                    display_mgr=display_mgr,
                )


def process_validation(cfg: Config, display_mgr: DisplayManager) -> None:
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    suffix = label_suffix_for_region(cfg.region)

    for person_entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        folder_person_id = person_entry.name
        for file_entry in sorted(os.scandir(person_entry.path), key=lambda e: e.name):
            if not file_entry.is_file() or not file_entry.name.lower().endswith((".avi", ".mp4")):
                continue

            video_stem = os.path.splitext(file_entry.name)[0]
            person_id, condition, state = parse_validation_or_test_video_name(video_stem)
            if person_id != folder_person_id:
                raise ValueError(f"Subject mismatch: {file_entry.path}")

            label_path = os.path.join(person_entry.path, f"{person_id}_{condition}_mixing_{suffix}.txt")
            if not os.path.isfile(label_path):
                msg = f"Missing validation label: {label_path}"
                if cfg.fail_if_label_missing:
                    raise FileNotFoundError(msg)
                print(f"[SKIP] {msg}")
                continue

            out_condition = build_output_condition(condition, cfg)
            output_dir = os.path.join(cfg.save_root, person_id, out_condition, state)
            os.makedirs(output_dir, exist_ok=True)
            save_prefix = f"{person_id}_{out_condition}_{state}"

            process_video(
                video_path=file_entry.path,
                label_path=label_path,
                output_dir=output_dir,
                model_path=cfg.model_path,
                save_prefix=save_prefix,
                region=cfg.region,
                padding_ratio=cfg.padding_ratio,
                display_name=video_stem,
                display_mgr=display_mgr,
            )


def process_test(cfg: Config, display_mgr: DisplayManager) -> None:
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    label_root = os.path.join(cfg.root_dir, "test_label_txt", "wh")
    if not os.path.isdir(label_root):
        raise FileNotFoundError(f"Test label folder not found: {label_root}")

    for entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not entry.is_file() or not entry.name.lower().endswith((".avi", ".mp4")):
            continue

        video_stem = os.path.splitext(entry.name)[0]
        person_id, condition, state = parse_validation_or_test_video_name(video_stem)
        label_path = os.path.join(label_root, f"{person_id}_{condition}_mixing_drowsiness.txt")

        if not os.path.isfile(label_path):
            msg = f"Missing test label: {label_path}"
            if cfg.fail_if_label_missing:
                raise FileNotFoundError(msg)
            print(f"[SKIP] {msg}")
            continue

        out_condition = build_output_condition(condition, cfg)
        output_dir = os.path.join(cfg.save_root, person_id, out_condition, state)
        os.makedirs(output_dir, exist_ok=True)
        save_prefix = f"{person_id}_{out_condition}_{state}"

        process_video(
            video_path=entry.path,
            label_path=label_path,
            output_dir=output_dir,
            model_path=cfg.model_path,
            save_prefix=save_prefix,
            region=cfg.region,
            padding_ratio=cfg.padding_ratio,
            display_name=video_stem,
            display_mgr=display_mgr,
        )


def run(cfg: Config):
    os.makedirs(cfg.save_root, exist_ok=True)
    display_mgr = DisplayManager(enabled=cfg.use_live_display)
    display_mgr.init()

    try:
        split = cfg.split.lower()
        if split == "train":
            process_train(cfg, display_mgr)
        elif split in {"validation", "val"}:
            process_validation(cfg, display_mgr)
        elif split == "test":
            process_test(cfg, display_mgr)
        else:
            raise ValueError(f"Unsupported split: {cfg.split}")
    finally:
        display_mgr.clear_live()


if __name__ == "__main__":
    CONFIG = Config(
        model_path="/content/face_landmarker.task",
        root_dir="/kaggle/input/dddeeee",
        save_root="/content/output",
        region="face",          # change here: face | left_eye | right_eye | mouth
        split="train",          # change here: train | validation | test
        padding_ratio=0.10,      # used for face / eye square crops
        use_live_display=True,
        fail_if_label_missing=True,
        normalize_output_condition=True,
    )

    run(CONFIG)
